In [1]:
# פרטי הסטודנטים:
# עמית ה - 4997
# עמית כ - 5261

In [2]:

# פרומפטים (AI):

# נעשה שימוש במודל שפה לבניית השלד של אלגוריתם ה-KNN מאפס בעזרת Numpy, 

# ולבניית ה-Pipeline של הנדסת המאפיינים (TF-IDF) וביצוע Grid Search.

# "תעזור לי לכתוב מחלקה של מודל KNN בפייתון מאפס בלי להשתמש ב-sklearn, רק עם NumPy. תדאג שפונקציית החיזוי תשתמש בווקטוריזציה כדי לרוץ מהר בלי לולאות for, ותוסיף אפשרות לבחור בין מרחק אוקלידי למרחק קוסינוס."

# "איך אני בונה Pipeline בפייתון שמחבר בין TfidfVectorizer למודל KNN? אני רוצה להריץ עליו GridSearchCV כדי למצוא את הפרמטרים הכי טובים, תוך שימוש ב-StratifiedKFold של 5 חלוקות שישמור על יחס הספאם בדאטה."

# "איך אפשר לחלץ את המילים הכי חשובות מתוך ה-TF-IDF אחרי שה-Grid Search סיים לאמן את המודל? אני רוצה להדפיס שתי רשימות נפרדות: 10 המילים שהכי מאפיינות ספאם, ו-10 המילים שהכי מאפיינות הודעות רגילות (Ham)."

# "יש לי דאטהסט של הודעות טקסט (SMS) לזיהוי ספאם. הבעיה היא שיש חוסר איזון קיצוני - יש כמעט 4,000 הודעות רגילות ורק 600 הודעות ספאם. איך כדאי לטפל בזה לפני האימון של המודל?"

# "הצעת לי להשתמש בשיטות כמו SMOTE (יצירת דוגמאות סינתטיות), אבל זה פחות מתאים פה כי מדובר בטקסט לפני וקטוריזציה. במקום זה, תכתוב לי קוד פשוט שמשתמש ב-RandomUnderSampler של ספריית imblearn כדי לחתוך את כמות ההודעות הרגילות ל-600, ותראה לי איך לשמור על המבנה של המערך כדי שלא יהיו שגיאות אחר כך ב-Pipeline."

In [3]:
# הסבר על ה-Dataset:
# בחרנו ב-Dataset של Kaggle לסיווג הודעות SMS כ-Spam (דואר זבל) או Ham (הודעה לגיטימית).
# זוהי בעיית סיווג בינארית (NLP). המטרה היא ללמד מודל לזהות דפוסים ומילים שמאפיינים הודעות זבל.

# Dataset Explanation:
# We chose a Kaggle dataset for classifying SMS messages as
# Spam (non legitimate messages) or Ham (legitimate messages).
# This is a binary classification problem (NLP).
# The goal is to train a model to identify 
# patterns and words that characterize spam messages.


In [4]:
# חלק 1 – הקדמה וטעינת נתונים

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, precision_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler

# טעינת ה-dataset
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'text'])

# המרת התוויות למספרים (0 = Ham, 1 = Spam)
df['label_num'] = df.label.map({'ham':0, 'spam':1})

# חלוקה ל-Train ו-Test
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df['text'], df['label_num'], test_size=0.2, random_state=42)

print("--- First 5 rows of Train-set ---")
print(X_train_raw.head())




--- First 5 rows of Train-set ---
1978    Reply to win £100 weekly! Where will the 2006 ...
3989    Hello. Sort of out in town already. That . So ...
3935     How come guoyang go n tell her? Then u told her?
4078    Hey sathya till now we dint meet not even a si...
4086    Orange brings you ringtones from all time Char...
Name: text, dtype: object


In [5]:
# חלק 2 - Feature Engineering

# המרת טקסט לווקטורים מספריים בעזרת TF-IDF, תוך הסרת מילות קישור (stop words)
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)

X_train = vectorizer.fit_transform(X_train_raw).toarray()
X_test = vectorizer.transform(X_test_raw).toarray()

print("\n--- Feature engineering on first 2 examples ---")
print(X_train[:2])




--- Feature engineering on first 2 examples ---
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [6]:
# חלק 3 - מימוש אלגוריתם למידה (Custom KNN)

class CustomKNN:
    def __init__(self, k=3, use_cosine=True):
        self.k = k
        self.use_cosine = use_cosine # הוספת פרמטר שמאפשר לבחור את שיטת חישוב המרחק
        
    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        
        # חישוב הנורמות מראש פעם אחת כדי לחסוך זמן חישוב בשלב החיזוי
        if self.use_cosine:
            self.norm_X_train = np.linalg.norm(self.X_train, axis=1)
        
    def predict(self, X):
        return np.array([self._predict(x) for x in X])
        
    def _predict(self, x):
        if self.use_cosine:
            # --- מרחק קוסינוס (Cosine Distance) ---
            # חישוב מכפלה סקלרית של הדוגמה החדשה עם כל מטריצת האימון בבת אחת
            dot_product = np.dot(self.X_train, x)
            norm_x = np.linalg.norm(x)
            
            # חישוב דמיון קוסינוס (הוספת ערך קטן מאוד כדי למנוע חלוקה באפס)
            similarity = dot_product / (self.norm_X_train * norm_x + 1e-10)
            
            # המרה למרחק (מרחק קטן יותר = דמיון גדול יותר)
            distances = 1 - similarity
            
        else:
            # --- מרחק אוקלידי (Euclidean Distance) בחישוב וקטורי ---
            # החלפנו את לולאת ה-for בחישוב של NumPy על כל המטריצה
            distances = np.linalg.norm(self.X_train - x, axis=1)
            
        # מציאת k האינדקסים של השכנים הקרובים ביותר
        k_indices = np.argsort(distances)[:self.k]
        
        # ספירת המחלקות והחזרת המחלקה הנפוצה ביותר
        return Counter(self.y_train[k_indices]).most_common(1)[0][0]


In [7]:
# # חלק 4 ו-5 - אימון, חיזוי ושערוך המודל

# אימון המודל המשופר (עם בחירה במרחק קוסינוס)
knn_model = CustomKNN(k=3, use_cosine=True)
knn_model.fit(X_train, y_train)

# חיזוי 5 הסיווגים הראשונים 
y_pred_5 = knn_model.predict(X_test[:5])
print("\n--- First 5 predictions on Test-set ---")
print("Predicted:", y_pred_5)
print("Actual:   ", y_test[:5].values)

# שערוך איכות המודל על *כל* ה-Test Set
y_pred_all = knn_model.predict(X_test)
f1 = f1_score(y_test, y_pred_all)
print(f"\nOptimized Custom KNN F1 Score (Spam class) on FULL test set: {f1:.3f}")




--- First 5 predictions on Test-set ---
Predicted: [0 0 0 0 0]
Actual:    [0 0 0 0 0]

Optimized Custom KNN F1 Score (Spam class) on FULL test set: 0.796


In [8]:
# # חלק 6 – סעיפי בונוס (הרחבות)

# 6.ד: הרחבות נוספות - התאמות מיוחדות של ה-Data (טיפול ב-Imbalanced Data)

print("--- 6.D: Handling Imbalanced Data (Under-Sampling) ---")
print(f"Original class distribution: {Counter(y_train)}")

# המרת הטקסט למבנה דו-ממדי שמתאים ל-imblearn
X_train_arr = np.array(X_train_raw).reshape(-1, 1)

# ביצוע Under-sampling: הקטנת כמות הודעות ה-Ham כך שתשווה לכמות ה-Spam
rus = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train_arr, y_train)

# החזרה למבנה של רשימת טקסטים
X_train_balanced = X_train_resampled.flatten()
print(f"Resampled class distribution: {Counter(y_train_resampled)}\n")


# 6.א + 6.ב + 6.ג + 6.ז: Grid Search, K-Fold CV, Hyperparameters & Feature Eng

print("--- Running Advanced Grid Search with Pipeline ---")

# יצירת ה-Pipeline שמאגד את הנדסת המאפיינים (TF-IDF) ואת מודל ה-KNN
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('knn', KNeighborsClassifier())
])

# רשת הפרמטרים: בודקים גם Feature Engineering וגם Hyperparameters של המודל
param_grid = {
    'tfidf__max_features': [500, 1000],          # התנסות בגודל אוצר המילים
    'tfidf__ngram_range': [(1, 1), (1, 2)],      # התנסות במילים בודדות מול צמדי מילים
    'knn__n_neighbors': [3, 5, 7, 9],            # התנסות במספר שכנים
    'knn__weights': ['uniform', 'distance'],     # התנסות במשקל המרחק
    'knn__metric': ['cosine', 'euclidean']       # התנסות בפונקציית המרחק
}

# חלוקה שומרת-יחס (Stratified) ל-5 קבוצות
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# הגדרת ה-Grid Search והרצתו על הנתונים המאוזנים (n_jobs=-1 משתמש בכל ליבות המעבד לחישוב מהיר)
grid_search = GridSearchCV(pipeline, param_grid, cv=skf, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train_balanced, y_train_resampled)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV F1 Macro Score: {grid_search.best_score_:.3f}\n")

# הצגת כל התוצאות ב-DataFrame (מסודר מהציון הגבוה לנמוך)
results_df = pd.DataFrame(grid_search.cv_results_).sort_values(by='rank_test_score')
print("Top 5 Grid Search Results:")
print(results_df[['param_tfidf__ngram_range', 'param_knn__n_neighbors', 'param_knn__metric', 'mean_test_score']].head())


# 6.ה: הרחבות נוספות - בדיקת איכות מיוחדת (Precision & Confusion Matrix)

print("\n--- 6.E: Special Quality Metrics on Test Set ---")

# שימוש במודל הטוב ביותר שנמצא כדי לחזות על נתוני הבדיקה
y_pred_best = grid_search.predict(X_test_raw)

# בבעיות ספאם, מדד ה-Precision הוא קריטי כדי לוודא שלא נשלח הודעות אמיתיות לספאם (False Positives)
precision = precision_score(y_test, y_pred_best)
print(f"Precision Score for Spam: {precision:.3f}")
print("Meaning: Out of all messages the model flagged as Spam, how many were ACTUALLY Spam?")

cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix (Ham=0, Spam=1):")
print(pd.DataFrame(cm, index=['Actual Ham', 'Actual Spam'], columns=['Predicted Ham', 'Predicted Spam']))


# 6.ו: Explainability - הבנת התוצאות והמאפיינים החשובים

print("\n--- 6.F: Explainability: Top Feature Importances (Separated by Class) ---")

# חילוץ ה-TF-IDF הטוב ביותר מתוך ה-Grid Search
best_tfidf = grid_search.best_estimator_.named_steps['tfidf']
feature_names = np.array(best_tfidf.get_feature_names_out())

# יצירת מטריצת ה-TF-IDF האופטימלית
X_train_transformed = best_tfidf.transform(X_train_balanced).toarray()

# הפרדת הנתונים
spam_indices = (y_train_resampled == 1)
ham_indices = (y_train_resampled == 0)

# חישוב משקל ה-TF-IDF הממוצע לכל מחלקה
mean_tfidf_spam = X_train_transformed[spam_indices].mean(axis=0)
mean_tfidf_ham = X_train_transformed[ham_indices].mean(axis=0)

# מציאת 10 המילים המובילות
top_spam_indices = np.argsort(mean_tfidf_spam)[-10:]
top_ham_indices = np.argsort(mean_tfidf_ham)[-10:]

print("Top terms indicating SPAM:")
print(feature_names[top_spam_indices])

print("\nTop terms indicating HAM (Legitimate):")
print(feature_names[top_ham_indices])

--- 6.D: Handling Imbalanced Data (Under-Sampling) ---
Original class distribution: Counter({0: 3859, 1: 598})
Resampled class distribution: Counter({0: 598, 1: 598})

--- Running Advanced Grid Search with Pipeline ---
Best Parameters: {'knn__metric': 'cosine', 'knn__n_neighbors': 3, 'knn__weights': 'distance', 'tfidf__max_features': 1000, 'tfidf__ngram_range': (1, 2)}
Best CV F1 Macro Score: 0.926

Top 5 Grid Search Results:
  param_tfidf__ngram_range  param_knn__n_neighbors param_knn__metric  \
7                   (1, 2)                       3            cosine   
4                   (1, 1)                       3            cosine   
6                   (1, 1)                       3            cosine   
3                   (1, 2)                       3            cosine   
0                   (1, 1)                       3            cosine   

   mean_test_score  
7         0.926183  
4         0.925369  
6         0.922046  
3         0.911899  
0         0.911082  

--- 6.E: S